## Import lib

In [1]:

import pandas as pd 
import numpy as np 
import os
import sys
sys.path.append('../')
sys.path.append('../..')
import random

from detection.lof import LocalOutlierFactorDetector
from sklearn.metrics import classification_report, confusion_matrix \
        , roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from preprocessing.load_data import DataLoader

## Read data

In [2]:


card_data = pd.read_csv('../data/raw/creditcard.csv')

print("Card data shape: ", card_data.shape) 

print(card_data.head(2))


data_use = DataLoader(data=card_data, label_col='Class')

data_use.remove_columns(['Time'])

Card data shape:  (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   

        V26       V27       V28  Amount  Class  
0 -0.189115  0.133558 -0.021053  149.62      0  
1  0.125895 -0.008983  0.014724    2.69      0  

[2 rows x 31 columns]


## Preprocessing

In [3]:

training, label = data_use.get_features(), data_use.get_labels()

print("Training data shape: ", training.shape) 

print(training.head(2))

print("label data shape: ", label.shape) 

print(label.head(2))

X_train, X_test, y_train, y_test = data_use.train_test_split(test_size=0.3, random_state=42)

print("X_train shape: ", X_train.shape) 
print("X_test shape: ", X_test.shape)

print("Sample X_train data: ", X_train.head(2))

Training data shape:  (284807, 29)
         V1        V2        V3        V4        V5        V6        V7  \
0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9       V10  ...       V20       V21       V22       V23  \
0  0.098698  0.363787  0.090794  ...  0.251412 -0.018307  0.277838 -0.110474   
1  0.085102 -0.255425 -0.166974  ... -0.069083 -0.225775 -0.638672  0.101288   

        V24       V25       V26       V27       V28  Amount  
0  0.066928  0.128539 -0.189115  0.133558 -0.021053  149.62  
1 -0.339846  0.167170  0.125895 -0.008983  0.014724    2.69  

[2 rows x 29 columns]
label data shape:  (284807,)
0    0
1    0
Name: Class, dtype: int64
X_train shape:  (199364, 29)
X_test shape:  (85443, 29)
Sample X_train data:                V1        V2        V3        V4        V5        V6        V7  \
2557   -2.289565 -0.480260  0.818685 -1.706423  0.822102 -1.66

## Model training

In [4]:

from detection import isolation_forest


print("Start training LOF model...")
lof = LocalOutlierFactorDetector(n_neighbors=20, contamination=0.01, novelty=True)

lof.fit(X_train)

print("Model training completed.")

print("Scoring test data...")

lof_scores = lof.score(X_test)

lof_predictions = lof.predict(X_test)

print("Local Outlier Factor Scores: ", lof_scores[:5])
print("Local Outlier Factor Predictions: ", lof_predictions[:5])


Start training LOF model...
Model training completed.
Scoring test data...


d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


Local Outlier Factor Scores:  [-6.2122221  -1.02452058 -1.03713291 -1.12147299 -0.98867367]
Local Outlier Factor Predictions:  [ True False False False False]


In [5]:
len(lof_scores)

85443

In [6]:
len(X_train)

199364

In [7]:
len(X_test)

85443

In [8]:
lof_scores

array([-6.2122221 , -1.02452058, -1.03713291, ..., -1.09401565,
       -0.97739906, -1.25756785], shape=(85443,))

In [9]:

print("Evaluating model performance...")

if y_test is not None:
    print("Number of anomalies detected: ", np.sum(lof_predictions))
    print("Percentage of anomalies detected: ", np.mean(lof_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, lof_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(y_test, lof_predictions.astype(int)))

    print("AUC-ROC of local outlier factor: ", roc_auc_score(y_test, lof_scores))

    if not os.path.exists('../models/checkpoint/'):
        os.makedirs('../models/checkpoint/')

    result_folder = '../models/checkpoint/baseline_lof/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    lof.save_model(result_folder + 'lof_baseline_model.pkl')

    classification_report_df = classification_report(y_test, lof_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'lof_classification_baseline_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(lof_predictions))

Evaluating model performance...
Number of anomalies detected:  851
Percentage of anomalies detected:  0.9959856278454643 %
Confusion Matrix:
[[84501   806]
 [   91    45]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     85307
           1       0.05      0.33      0.09       136

    accuracy                           0.99     85443
   macro avg       0.53      0.66      0.54     85443
weighted avg       1.00      0.99      0.99     85443

AUC-ROC of local outlier factor:  0.27830124277781493
